In [ ]:
import grewpy
from grewpy import Corpus, CorpusDraft, Request
from collections import Counter
import sys
sys.path.insert(1, "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/tod")

import tod.corpus
import tod.outliers
import tod.clustering
import tod.plotting
import tod.dimension_reduction_classic

# treebank_path = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/data/input/Universal_Dependencies/ud-treebanks-v2.15/UD_French-GSD"
treebank_path = "/Users/madalina/Downloads/bUD_English-GUM"
grew_pattern = "pattern{X[upos<>PUNCT]}"
patterns_text_file = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/scripts/3. probability_matrix/patterns_all_nodes.txt"
analysed_category = "all_nodes"

corpus = tod.corpus.Corpus(
    treebank_path=treebank_path,
    grew_pattern=grew_pattern,
    patterns_text_file=patterns_text_file,
    use_sud=False,
    matrix_type="coverage",
    # excluded_feature_patterns=[r"CxnElt=", r"Cxn=", r"own", r"Gender"]
        excluded_feature_patterns=[r"CxnElt=", r"Cxn=", r"XML=", r"PDTB=", r"SplitAnte=", r"MSeg=", r"Entity=", r"Discourse=", r"Bridge=", r"own"]
)

In [267]:
import numpy as np
from scipy.spatial.distance import pdist, squareform

x = np.array([[2, 0, 2], [2, 2, 3], [-2, 4, 5], [0, 1, 9], [2, 2, 4]])
# x = np.array([[2, 2, 4], [-2, 4, 5]])

y = pdist(x)
squareform(y)

array([[0.        , 2.23606798, 6.40312424, 7.34846923, 2.82842712],
       [2.23606798, 0.        , 4.89897949, 6.40312424, 1.        ],
       [6.40312424, 4.89897949, 0.        , 5.38516481, 4.58257569],
       [7.34846923, 6.40312424, 5.38516481, 0.        , 5.47722558],
       [2.82842712, 1.        , 4.58257569, 5.47722558, 0.        ]])

In [303]:
# corpus.lexunit2idx(("pas", "ADV"))
corpus.lexunit2idx(("divers", "DET"))
# corpus.lexunit2idx(("jamais", "ADV"))
# corpus.lexunit2idx(("have", "AUX"))
# corpus.lexunit2idx(("from", "SCONJ"))

# corpus._lexunit2idx

1046

In [307]:
x = corpus.feature_matrix
y = squareform(pdist(x))
y

array([[0.        , 0.05211651, 0.37852579, ..., 0.17784524, 0.20291186,
        0.04817015],
       [0.05211651, 0.        , 0.3721046 , ..., 0.17653189, 0.20017149,
        0.05758609],
       [0.37852579, 0.3721046 , 0.        , ..., 0.34275402, 0.36200197,
        0.372227  ],
       ...,
       [0.17784524, 0.17653189, 0.34275402, ..., 0.        , 0.16350768,
        0.17214373],
       [0.20291186, 0.20017149, 0.36200197, ..., 0.16350768, 0.        ,
        0.20953787],
       [0.04817015, 0.05758609, 0.372227  , ..., 0.17214373, 0.20953787,
        0.        ]])

In [283]:
np.argsort(y[1559])

array([1559, 1242,  918, ..., 1054, 1277, 1296])

In [284]:
corpus.idx2lexunit(1970)

('town', 'NOUN')

In [305]:
rows_minimatrix, columns_minimatrix = set(), set()
for f in corpus._feature2idx:
    rows_minimatrix.add(f.split(":")[2])
    columns_minimatrix.add(f.split(":")[-1])
rows_minimatrix = list(rows_minimatrix)
columns_minimatrix = list(columns_minimatrix)
print(rows_minimatrix)
print(columns_minimatrix)

['next', 'parent', 'child', 'prev']
['position=before', 'upos=SCONJ', 'rel_shallow=amod', 'Tense=Imp', 'relcl', 'Number__psor=Plur', 'PronType=Prs', 'rel_shallow=iobj', 'ExtPos=CCONJ', 'Mood=Ind', 'PronType=Neg', 'name', 'Tense=Pres', 'Polarity=Pos', 'Tense=Past', 'outer', 'Subject=SubjRaising', 'upos=PRON', 'rel_shallow=cop', 'mod', 'rel_shallow=dislocated', 'pass', 'PronType=Exc', 'ExtPos=DET', 'rel_shallow=mark', 'Mood=Sub', 'Subject=Generic', 'rel_shallow=parataxis', 'Definite=Ind', 'upos=INTJ', 'rel_shallow=nummod', 'upos=NOUN', 'Voice=Act', 'rel_shallow=xcomp', 'subj', 'rel_shallow=punct', 'rel_shallow=obj', 'Person=1', 'PronType=Rel', 'Tense=Fut', 'NumType=Ord', 'rel_shallow=vocative', 'upos=PUNCT', 'comp', 'upos=ADJ', 'Reflex=Yes', 'upos=SYM', 'rel_shallow=csubj', 'Polarity=Neg', 'rel_shallow=discourse', 'ExtPos=SCONJ', 'Subject=Instantiated', 'Subject=OblRaising', 'Person__psor=2', 'PronType=Int', 'rel_shallow=dep', 'upos=DET', 'Number__psor=Sing', 'Person__psor=3', 'arg', 're

In [309]:
mini_matrices = []
top_similar_indices = list(np.argsort(y[1046]))[:50]
# print([corpus.idx2lexunit(i) for i in top_similar_indices])
for i in top_similar_indices:
    mini_matrix = np.zeros((len(rows_minimatrix), len(columns_minimatrix)))
    current_word_row = list(x[i])
    for idx, value in enumerate(current_word_row):
        feature = corpus.idx2feature(idx)
        row = rows_minimatrix.index(feature.split(":")[2])
        column = columns_minimatrix.index(feature.split(":")[-1])
        mini_matrix[row, column] = x[i, idx]
    mini_matrices.append(mini_matrix)



In [310]:
diff = mini_matrices[0] - mini_matrices[1]
biggest_diffs = np.unravel_index(np.argsort(abs(diff), axis=None)[-5:], diff.shape)
print(biggest_diffs)
for i in range(5):
    print(rows_minimatrix[biggest_diffs[0][i]], columns_minimatrix[biggest_diffs[1][i]], diff[biggest_diffs[0][i], biggest_diffs[1][i]])

(array([0, 3, 3, 3, 3]), array([76, 98, 42, 90, 73]))
next Number=Plur 0.005036705402113456
prev VerbForm=Inf -0.0060901339829476245
prev upos=PUNCT -0.009210916153668893
prev upos=VERB -0.014083023340026993
prev upos=ADP 0.05543009513776871


In [311]:
diff[1,64]

0.0

In [312]:
diff[biggest_diff]

0.0

In [313]:
print(rows_minimatrix[biggest_diff[0]])
print(columns_minimatrix[biggest_diff[1]])

prev
Poss=Yes


In [314]:
complementarity_scores = {}
for i in top_similar_indices:
    if i == top_similar_indices[0]:
        highest_feats = np.argsort(x[top_similar_indices[0]])[::-1][:5].tolist()
        complementarity_scores[corpus.idx2lexunit(i)] = {"score": 0, "features": [corpus.idx2feature(idx) for idx in highest_feats]}
        continue
    diff = x[top_similar_indices[0]] - x[i]
    
    complementarity_scores[corpus.idx2lexunit(i)]= {}
    complementarity_scores[corpus.idx2lexunit(i)]["score"] = np.sum(abs(diff))
    highest_diff_feats_idx = np.argsort(np.abs(diff))[::-1][:5].tolist()
    # print(highest_diff_feats_idx)
    complementarity_scores[corpus.idx2lexunit(i)]["features"] = [(corpus.idx2feature(idx), float(diff[idx])) for idx in highest_diff_feats_idx]


complementarity_scores

{('divers', 'DET'): {'score': 0,
  'features': ['node:X:parent:upos=NOUN',
   'node:X:parent:Number=Plur',
   'node:X:next:Number=Plur',
   'node:X:parent:position=after',
   'node:X:next:upos=NOUN']},
 ('certain', 'DET'): {'score': 0.1637949764624552,
  'features': [('node:X:prev:upos=ADP', 0.05543009513776871),
   ('node:X:prev:upos=VERB', -0.014083023340026993),
   ('node:X:prev:upos=PUNCT', -0.009210916153668893),
   ('node:X:prev:VerbForm=Inf', -0.0060901339829476245),
   ('node:X:next:Number=Plur', 0.005036705402113456)]},
 ('différent', 'DET'): {'score': 0.2580828019030267,
  'features': [('node:X:prev:upos=ADP', 0.0360967709282316),
   ('node:X:prev:upos=VERB', -0.02081182305901407),
   ('node:X:prev:Person=3', -0.017066504706954144),
   ('node:X:prev:VerbForm=Fin', -0.017066504706954144),
   ('node:X:prev:Mood=Ind', -0.017066504706954144)]},
 ('vingt', 'NUM'): {'score': 0.4135135135135135,
  'features': [('node:X:prev:upos=ADP', 0.04639639639639641),
   ('node:X:next:upos=NOUN

In [315]:
arr = np.array([[1, 2, 3], [4, 5, 6]])
np.sum(arr)

21

In [316]:
import plotly.graph_objects as go
import numpy as np

def plot_complementarity_with_axis(comp_scores: dict, top_k_features: int = 5):
    # --- 1. Data Preparation (Same as before) ---
    items = list(comp_scores.items())
    rest = sorted(items[1:], key=lambda kv: kv[1]['score'])
    ordered = [items[0]] + rest

    def fmt_key(k):
        lemma, upos = k
        return f"<b>{lemma}</b><br>({upos})"

    def fmt_features_hover(feats):
        if not feats: return ""
        if isinstance(feats[0], tuple):
            lines = [f"{f}: {v:+.3f}" for f, v in feats[:top_k_features]]
        else:
            lines = [str(f) for f in feats[:top_k_features]]
        return "<br>".join(lines)
    
    labels = [fmt_key(k) for k, _ in ordered]
    xvals = [0.0] + [v['score'] for _, v in ordered[1:]]
    yvals = np.zeros(len(ordered))
    
    hovertexts = []
    for i, (k, v) in enumerate(ordered):
        feat_str = fmt_features_hover(v['features'])
        score_str = f"{v['score']:.4f}" if i > 0 else "0 (Base)"
        hovertexts.append(f"{fmt_key(k)}<br>Score: {score_str}<br><br>Features:<br>{feat_str}")

    fig = go.Figure()

    # --- 2. Draw the Points ---
    fig.add_trace(go.Scatter(
        x=xvals, y=yvals,
        mode="markers",
        marker=dict(size=14, color='#2c3e50', line=dict(width=2, color='white')), # Added white border for contrast
        hovertext=hovertexts,
        hoverinfo="text",
        showlegend=False,
        name="Words"
    ))

    # --- 3. Draw the Axis Line (The Fix) ---
    # We draw a line shape from slightly before the first point to slightly after the last
    min_x, max_x = min(xvals), max(xvals)
    padding = (max_x - min_x) * 0.05 # 5% padding
    
    fig.add_shape(
        type="line",
        x0=min_x - padding, y0=0,
        x1=max_x + padding, y1=0,
        line=dict(color="black", width=3),
        layer="below" # Ensures the line is BEHIND the dots
    )

    # --- 4. Staggered Annotations ---
    stagger_offsets = [-40, 40, -90, 90, -140, 140]
    
    for i, (xi, lbl) in enumerate(zip(xvals, labels)):
        offset = stagger_offsets[i % len(stagger_offsets)]
        y_anchor = "bottom" if offset < 0 else "top"
        
        # Add a subtle vertical connector line from the axis to the text
        # This helps visually connect floating text back to the axis line
        fig.add_shape(
            type="line",
            x0=xi, y0=0,
            x1=xi, y1=0, # The annotation arrow handles the rest, but you can draw a full line if needed
            line=dict(color="#ddd", width=1, dash="dot"),
            layer="below"
        )

        fig.add_annotation(
            x=xi, y=0,
            text=lbl,
            showarrow=True,
            arrowhead=0,
            arrowwidth=1.5,
            arrowcolor="#555",
            ax=0, 
            ay=offset,
            font=dict(size=11),
            bgcolor="rgba(255,255,255,0.9)",
            bordercolor="#ddd",
            borderwidth=1,
            borderpad=4
        )

    # --- 5. Clean Layout ---
    fig.update_layout(
        title="Complementarity Distance",
        xaxis_title="Score",
        yaxis=dict(visible=False, range=[-1, 1]),
        xaxis=dict(
            visible=True, 
            showgrid=False, 
            zeroline=False, # We drew our own line, so turn off the default
            showticklabels=True
        ),
        margin=dict(t=100, b=100, l=50, r=50),
        height=600,
        width=1500,
        plot_bgcolor="white"
    )

    return fig

fig = plot_complementarity_with_axis(complementarity_scores)
fig.show()

In [317]:
fig.write_html("divers.html")